# Benchmark Retrieval Strategies with KB Arena before wiring into Haystack

[KB Arena](https://github.com/xmpuspus/kb-arena) is an open-source benchmark that runs nine architecturally distinct retrieval strategies head-to-head on your own documentation and reports IR metrics with statistical confidence intervals.

Before you commit to a Haystack pipeline using one specific retrieval approach, KB Arena lets you ask: *for my corpus, which strategy actually wins?* This notebook walks through running the benchmark on a small example corpus, reading the results, and wiring the winning strategy into a Haystack `Pipeline`.

Strategies covered: naive vector, contextual vector, QnA pairs, knowledge graph (Neo4j), hybrid (RRF-fused), RAPTOR, PageIndex, BM25, rerank-vector (cross-encoder). The full benchmark requires Neo4j and embedding-provider API keys; this notebook uses BM25 only so it runs end-to-end with no external services.

Notebook by [*Xavier Puspus*](https://github.com/xmpuspus)

## Prerequisites

No API keys are required for the BM25-only path shown here. To unlock the full nine-strategy benchmark, see the KB Arena [README](https://github.com/xmpuspus/kb-arena) for setting `KB_ARENA_ANTHROPIC_API_KEY` (or any other supported provider) and `KB_ARENA_NEO4J_URI`.

## Install dependencies

In [ ]:
!pip install kb-arena haystack-ai

## Step 1 — Prepare a small example corpus

KB Arena ingests Markdown, HTML, PDF, DOCX, plaintext, CSV, and a few other formats from a directory. For this walkthrough we drop three short AWS-flavored Markdown files into a fresh directory.

In [ ]:
from pathlib import Path

corpus_dir = Path("raw")
corpus_dir.mkdir(exist_ok=True)

(corpus_dir / "lambda.md").write_text("""# AWS Lambda

AWS Lambda runs code in response to events without provisioning servers. It scales
automatically and you pay only for the compute time consumed. Common triggers
include S3 uploads, DynamoDB streams, API Gateway requests, and scheduled
EventBridge rules. Memory is configurable from 128 MB to 10,240 MB; CPU scales
with memory. The maximum execution timeout is 15 minutes per invocation.
""")

(corpus_dir / "ec2.md").write_text("""# Amazon EC2

Amazon EC2 provides resizable compute capacity in the cloud through virtual
machines called instances. Instance types are grouped into families optimized
for compute, memory, storage, or accelerated computing. Pricing models include
On-Demand, Reserved, Savings Plans, and Spot. EBS volumes provide persistent
block storage attached to instances; instance store provides ephemeral local
storage.
""")

(corpus_dir / "fargate.md").write_text("""# AWS Fargate

AWS Fargate is a serverless compute engine for containers that works with both
Amazon ECS and Amazon EKS. You define container images, CPU and memory, and
Fargate provisions the underlying infrastructure for you. Tasks can run for as
long as needed and you pay per second of vCPU and memory consumed. Unlike
Lambda, Fargate has no 15-minute execution cap.
""")

print(sorted(p.name for p in corpus_dir.iterdir()))

## Step 2 — Initialize the corpus and ingest documents

KB Arena treats every corpus as an isolated workspace under `datasets/<corpus>/`. `init-corpus` creates the directory layout; `ingest` parses the source files into a JSONL `Document` representation.

In [ ]:
!kb-arena init-corpus aws-mini
!cp raw/*.md datasets/aws-mini/raw/
!kb-arena ingest datasets/aws-mini/raw/ --corpus aws-mini

## Step 3 — Generate benchmark questions

For real corpora, KB Arena auto-generates questions across five difficulty tiers via `kb-arena generate-questions`. For this minimal walkthrough we hand-author a tiny `questions.yaml` so the notebook stays fully offline.

In [ ]:
questions_yaml = '''questions:
  - id: q1
    text: What is the maximum execution timeout for an AWS Lambda invocation?
    tier: 1
    expected_docs: [lambda]
  - id: q2
    text: Which pricing models does Amazon EC2 support?
    tier: 1
    expected_docs: [ec2]
  - id: q3
    text: How does Fargate differ from Lambda for long-running container workloads?
    tier: 3
    expected_docs: [fargate, lambda]
'''

Path("datasets/aws-mini/questions").mkdir(parents=True, exist_ok=True)
Path("datasets/aws-mini/questions/questions.yaml").write_text(questions_yaml)

## Step 4 — Run KB Arena's retriever-lab on BM25

`retriever-lab` is the retrieval-only benchmark (no generation, no LLM judge). It computes IR metrics per question and aggregates with paired-bootstrap 95% confidence intervals.

In [ ]:
!kb-arena retriever-lab --corpus aws-mini --strategies bm25 --top-ks 3,5

## Step 5 — Inspect the IR metrics

Results land in `results/run_<id>/retriever_lab.json`. The per-strategy summary carries Recall@k, Precision@k, MRR, NDCG (binary + graded), MAP, R-Precision, and bpref, plus bootstrap CIs.

In [ ]:
import json
from pathlib import Path

run_dir = sorted(Path("results").glob("run_*"))[-1]
summary = json.loads((run_dir / "retriever_lab.json").read_text())

for strategy, metrics in summary.get("per_strategy", {}).items():
    print(f"\n{strategy}")
    for key in ("recall_at_k", "ndcg_at_k", "mrr", "map"):
        if key in metrics:
            print(f"  {key}: {metrics[key]}")

## Step 6 — Wire the winning strategy into a Haystack pipeline

Once KB Arena tells you which strategy wins on your corpus (with statistical confidence, not just mean delta), reach for the equivalent Haystack component. BM25 in KB Arena maps to `InMemoryBM25Retriever` here.

In [ ]:
from haystack import Document, Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever

docs = [Document(content=p.read_text(), meta={"name": p.stem}) for p in corpus_dir.iterdir()]
store = InMemoryDocumentStore()
store.write_documents(docs)

pipe = Pipeline()
pipe.add_component("retriever", InMemoryBM25Retriever(document_store=store, top_k=3))

result = pipe.run({"retriever": {"query": "What is the maximum execution timeout for an AWS Lambda invocation?"}})
for doc in result["retriever"]["documents"]:
    print(f"{doc.meta['name']}  score={doc.score:.3f}")

## Next steps

- Replace the three Markdown files with your real corpus, drop them in `datasets/<your-corpus>/raw/`, and re-run.
- Run `kb-arena generate-questions --corpus <your-corpus> --count 50` to auto-generate questions across five difficulty tiers.
- Run `kb-arena label-chunks --corpus <your-corpus>` to get chunk-level ground truth via BM25 + Haiku judge.
- Add embedding and Neo4j configuration to compare BM25 against vector, knowledge graph, hybrid, RAPTOR, PageIndex, and rerank strategies.
- Use `kb-arena optimize` to search across strategies and top-k values with bootstrap CIs, Wilcoxon p-values, and a Pareto frontier across (NDCG, latency).